# MOSAIC Runtime Consistency Benchmark

This notebook evaluates exact shared-prefix reuse on unmodified causal language models. It measures full prefill, ideal prefix reuse, materialized Hugging Face cache branching, amortized execution, cache-key safety, and an optional vLLM Automatic Prefix Caching baseline.

Default model set: TinyLlama-1.1B, Qwen2.5-1.5B, and Mistral-7B.

In [ ]:
import os
import sys
import subprocess

PACKAGES = [
    "transformers>=4.57,<5",
    "accelerate>=1.10,<2",
    "bitsandbytes>=0.46,<1",
    "huggingface_hub>=0.34,<1",
    "safetensors>=0.6,<1",
    "sentencepiece>=0.2,<1",
    "pandas>=2.2,<3",
    "matplotlib>=3.8,<4",
    "psutil>=5.9,<8",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", *PACKAGES
])
print("Dependencies installed.")

In [ ]:
import copy
import gc
import hashlib
import inspect
import json
import math
import platform
import random
import statistics
import time
import traceback
import warnings
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn.functional as F
import transformers
from huggingface_hub import HfApi, login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DynamicCache,
)

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/mosaic_llm_runtime")
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU accelerator must be enabled.")

DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_BYTES = torch.cuda.get_device_properties(0).total_memory
print("GPU:", GPU_NAME, f"({GPU_BYTES / 2**30:.1f} GiB)")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

## Execution profiles

- `smoke`: short validation run
- `paper_t4`: repeated measurements on an NVIDIA T4
- `a100_7b`: 7B-class runs with 4-bit loading

Run the Hugging Face benchmark before enabling the optional vLLM comparison.

In [ ]:
RUN_PROFILE = "paper_t4"
RUN_VLLM = False

PROFILES = {
    "smoke": {
        "contexts": [256, 1024],
        "workloads": ["agent"],
        "suffixes_per_workload": 2,
        "repeats": 3,
        "warmups": 1,
        "stream_requests": 8,
        "models": "t4",
    },
    "paper_t4": {
        "contexts": [256, 512, 1024, 2048],
        "workloads": ["agent", "rag"],
        "suffixes_per_workload": 3,
        "repeats": 7,
        "warmups": 2,
        "stream_requests": 16,
        "models": "t4",
    },
    "a100_7b": {
        "contexts": [512, 1024, 2048, 4096],
        "workloads": ["agent", "rag", "repository"],
        "suffixes_per_workload": 3,
        "repeats": 8,
        "warmups": 2,
        "stream_requests": 16,
        "models": "7b",
    },
}

MODEL_SUITES = {
    "t4": [
        {
            "family": "Llama",
            "model_id": "meta-llama/Llama-3.2-1B-Instruct",
            "quantization": None,
            "gated": True,
        },
        {
            "family": "Qwen",
            "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
            "quantization": None,
            "gated": False,
        },
        {
            "family": "Mistral",
            "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
            "quantization": "bnb4",
            "gated": False,
        },
    ],
    "7b": [
        {
            "family": "Llama",
            "model_id": "meta-llama/Llama-3.1-8B-Instruct",
            "quantization": "bnb4",
            "gated": True,
        },
        {
            "family": "Qwen",
            "model_id": "Qwen/Qwen2.5-7B-Instruct",
            "quantization": "bnb4",
            "gated": False,
        },
        {
            "family": "Mistral",
            "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
            "quantization": "bnb4",
            "gated": False,
        },
    ],
}

CFG = PROFILES[RUN_PROFILE]
MODELS = MODEL_SUITES[CFG["models"]]
CONTEXTS = CFG["contexts"]
REPEATS = CFG["repeats"]
WARMUPS = CFG["warmups"]
STREAM_REQUESTS = CFG["stream_requests"]

RANDOM_SEED = 20260721
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(json.dumps(CFG, indent=2))
print("Models:", [m["model_id"] for m in MODELS])

In [ ]:
def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face login complete.")
else:
    print("HF_TOKEN not found. Gated Llama models may be skipped.")

## Repeated-prefix workloads

Each workload contains a shared static prefix and request-specific suffixes. Text is re-tokenized for each model and trimmed to the target token length.

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful repository and retrieval analysis agent. "
    "Preserve exact symbol names, cite paths, do not invent execution results, "
    "and separate verified facts from hypotheses."
)

TOOL_SCHEMA = json.dumps({
    "tools": [
        {
            "name": "search_repository",
            "description": "Search source files.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "path": {"type": "string"},
                    "limit": {"type": "integer"},
                },
                "required": ["query"],
            },
        },
        {
            "name": "run_tests",
            "description": "Run a test target.",
            "parameters": {
                "type": "object",
                "properties": {
                    "target": {"type": "string"},
                    "timeout": {"type": "integer"},
                },
                "required": ["target"],
            },
        },
    ]
}, indent=2)

RAG_DOCUMENTS = "\n\n".join([
    "Document A: Prefix caching reuses computation only when token sequences share an exact causal prefix.",
    "Document B: Cache keys must include model identity, tokenizer identity, and exact token content.",
    "Document C: A cache branch must not mutate the immutable shared prefix state.",
    "Document D: Floating-point reassociation may change low-order bits without changing the top prediction.",
    "Document E: End-to-end latency includes lookup, branch creation, scheduling, and suffix computation.",
] * 96)

REPOSITORY_TEXT = "\n\n".join([
    "src/runtime/cache.py: defines immutable cache keys and request-local branches.",
    "src/runtime/planner.py: identifies exact shared prefixes and estimates break-even reuse.",
    "src/runtime/executor.py: executes compiled prefix state followed by a private suffix.",
    "tests/test_cache.py: checks model fingerprinting, token equality, and branch isolation.",
    "benchmarks/prefill.py: measures cold compile, warm reuse, latency, throughput, and memory.",
] * 128)

WORKLOAD_PREFIXES = {
    "agent": SYSTEM_PROMPT + "\n\nAVAILABLE TOOLS:\n" + TOOL_SCHEMA,
    "rag": SYSTEM_PROMPT + "\n\nRETRIEVED CONTEXT:\n" + RAG_DOCUMENTS,
    "repository": SYSTEM_PROMPT + "\n\nREPOSITORY SNAPSHOT:\n" + REPOSITORY_TEXT,
}

SUFFIXES = [
    "\nUser: Explain the cache invalidation policy.\nAssistant:",
    "\nUser: Find the likely source of a branch mutation bug.\nAssistant:",
    "\nUser: Propose tests for exact-prefix correctness.\nAssistant:",
    "\nUser: Estimate when compilation becomes profitable.\nAssistant:",
    "\nUser: Compare cold and warm request behavior.\nAssistant:",
    "\nUser: Identify which claims require an end-to-end serving benchmark.\nAssistant:",
]

In [ ]:
def sync():
    torch.cuda.synchronize()

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    sync()

def model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return DEVICE

def parameter_count(model):
    return int(sum(p.numel() for p in model.parameters()))

def maximum_context(model):
    values = []
    for name in (
        "max_position_embeddings",
        "n_positions",
        "max_sequence_length",
        "seq_length",
    ):
        value = getattr(model.config, name, None)
        if isinstance(value, int) and value > 0:
            values.append(value)
    return min(values) if values else 4096

def filtered_forward_kwargs(model, kwargs):
    signature = inspect.signature(model.forward)
    return {
        key: value
        for key, value in kwargs.items()
        if key in signature.parameters and value is not None
    }

def cache_key(model_id, revision, tokenizer, token_ids):
    payload = {
        "model_id": model_id,
        "revision": revision,
        "tokenizer_class": tokenizer.__class__.__name__,
        "tokens": [int(x) for x in token_ids],
    }
    raw = json.dumps(payload, separators=(",", ":"), sort_keys=True).encode()
    return hashlib.sha256(raw).hexdigest()

def make_exact_prefix(tokenizer, text, target_tokens):
    unit = tokenizer.encode(text, add_special_tokens=False)
    if not unit:
        raise ValueError("Workload prefix tokenized to an empty sequence.")
    repeats = math.ceil(target_tokens / len(unit))
    return (unit * repeats)[:target_tokens]

def encode_suffix(tokenizer, text, max_tokens=32):
    ids = tokenizer.encode(text, add_special_tokens=False)
    if not ids:
        fallback = tokenizer.eos_token_id
        if fallback is None:
            raise ValueError("Suffix tokenized to empty and tokenizer has no EOS token.")
        ids = [fallback]
    return ids[:max_tokens]

## Immutable prefix snapshots

The prefix cache is stored as an immutable tensor snapshot. A fresh cache object is created for each request. The ideal measurement excludes branch preparation; the materialized measurement includes Hugging Face cache reconstruction. Full prefill uses `use_cache=True` so both paths produce KV state.

In [ ]:
def freeze_cache(past_key_values):
    if hasattr(past_key_values, "to_legacy_cache"):
        legacy = past_key_values.to_legacy_cache()
    else:
        legacy = past_key_values

    frozen = []
    for layer in legacy:
        if not isinstance(layer, (tuple, list)):
            raise TypeError(f"Unsupported cache layer type: {type(layer)}")
        frozen.append(tuple(t.detach().clone() for t in layer))
    return tuple(frozen)

def fresh_cache(model, frozen_legacy):
    try:
        return DynamicCache.from_legacy_cache(frozen_legacy)
    except Exception:
        try:
            cache = DynamicCache(config=model.config)
        except TypeError:
            cache = DynamicCache()
        for layer_idx, layer in enumerate(frozen_legacy):
            key_states, value_states = layer[:2]
            cache.update(key_states, value_states, layer_idx)
        return cache

def cache_bytes(frozen_legacy):
    seen = set()
    total = 0
    for layer in frozen_legacy:
        for tensor in layer:
            ptr = tensor.data_ptr()
            if ptr not in seen:
                total += tensor.numel() * tensor.element_size()
                seen.add(ptr)
    return int(total)

def logit_metrics(full_logits, cached_logits):
    full = full_logits.float()
    cached = cached_logits.float()
    delta = full - cached
    full_logp = F.log_softmax(full, dim=-1)
    cached_logp = F.log_softmax(cached, dim=-1)
    kl = (full_logp.exp() * (full_logp - cached_logp)).sum(dim=-1).mean()
    return {
        "max_abs_logit_error": float(delta.abs().max().item()),
        "mean_abs_logit_error": float(delta.abs().mean().item()),
        "cosine_similarity": float(
            F.cosine_similarity(full, cached, dim=-1).mean().item()
        ),
        "top1_agreement": float(
            (full.argmax(-1) == cached.argmax(-1)).float().mean().item()
        ),
        "kl_full_to_cached": float(kl.item()),
    }

In [ ]:
def percentile_ci(values, confidence=0.95, bootstrap_samples=2000):
    values = np.asarray(values, dtype=np.float64)
    if len(values) == 0:
        return (float("nan"), float("nan"))
    if len(values) == 1:
        return (float(values[0]), float(values[0]))
    rng = np.random.default_rng(RANDOM_SEED)
    means = np.empty(bootstrap_samples)
    for i in range(bootstrap_samples):
        means[i] = rng.choice(values, size=len(values), replace=True).mean()
    alpha = 1.0 - confidence
    return (
        float(np.quantile(means, alpha / 2)),
        float(np.quantile(means, 1 - alpha / 2)),
    )

def timed_cuda(fn, repeats, warmups):
    for _ in range(warmups):
        fn()
    sync()

    values = []
    peaks = []
    for _ in range(repeats):
        torch.cuda.reset_peak_memory_stats()
        sync()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        sync()
        values.append(float(start.elapsed_time(end)))
        peaks.append(int(torch.cuda.max_memory_allocated()))
    return values, peaks

def summarize_times(values):
    lo, hi = percentile_ci(values)
    return {
        "median_ms": float(np.median(values)),
        "mean_ms": float(np.mean(values)),
        "sd_ms": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
        "mean_ci95_low_ms": lo,
        "mean_ci95_high_ms": hi,
    }

In [ ]:
def load_model(spec):
    model_id = spec["model_id"]
    quantization = spec["quantization"]

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=HF_TOKEN,
        use_fast=True,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = {
        "token": HF_TOKEN,
        "low_cpu_mem_usage": True,
        "device_map": {"": 0},
    }

    if quantization == "bnb4":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["dtype"] = torch.float16

    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            attn_implementation="sdpa",
            **kwargs,
        )
    except Exception as first_error:
        print("SDPA load failed; retrying default attention:", repr(first_error))
        model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)

    model.eval()
    revision = None
    try:
        revision = HfApi().model_info(model_id, token=HF_TOKEN).sha
    except Exception:
        revision = "unknown"

    return tokenizer, model, revision

In [ ]:
@torch.inference_mode()
def compile_prefix(model, prefix_ids):
    device = model_device(model)
    prefix = torch.tensor(prefix_ids, dtype=torch.long, device=device).unsqueeze(0)
    attention_mask = torch.ones_like(prefix)
    positions = torch.arange(prefix.shape[1], device=device).unsqueeze(0)
    cache_position = positions[0]

    kwargs = filtered_forward_kwargs(model, {
        "input_ids": prefix,
        "attention_mask": attention_mask,
        "position_ids": positions,
        "cache_position": cache_position,
        "use_cache": True,
    })

    def call():
        return model(**kwargs)

    for _ in range(WARMUPS):
        call()
    sync()

    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    output = call()
    end.record()
    sync()

    compile_ms = float(start.elapsed_time(end))
    compile_peak = int(torch.cuda.max_memory_allocated())
    frozen = freeze_cache(output.past_key_values)
    return frozen, compile_ms, compile_peak

@torch.inference_mode()
def full_forward(model, full_ids):
    device = model_device(model)
    input_ids = torch.tensor(full_ids, dtype=torch.long, device=device).unsqueeze(0)
    attention_mask = torch.ones_like(input_ids)
    positions = torch.arange(input_ids.shape[1], device=device).unsqueeze(0)
    kwargs = filtered_forward_kwargs(model, {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "position_ids": positions,
        "cache_position": positions[0],
        "use_cache": True,
    })
    return model(**kwargs).logits[:, -1, :]

@torch.inference_mode()
def cached_forward(model, frozen, prefix_length, suffix_ids):
    device = model_device(model)
    suffix = torch.tensor(suffix_ids, dtype=torch.long, device=device).unsqueeze(0)
    total_length = prefix_length + suffix.shape[1]
    attention_mask = torch.ones(
        (1, total_length),
        dtype=torch.long,
        device=device,
    )
    positions = torch.arange(
        prefix_length,
        total_length,
        device=device,
    ).unsqueeze(0)
    cache = fresh_cache(model, frozen)
    kwargs = filtered_forward_kwargs(model, {
        "input_ids": suffix,
        "attention_mask": attention_mask,
        "position_ids": positions,
        "cache_position": positions[0],
        "past_key_values": cache,
        "use_cache": True,
    })
    return model(**kwargs).logits[:, -1, :]

In [ ]:
def benchmark_one_suffix(
    model,
    tokenizer,
    frozen,
    prefix_ids,
    suffix_ids,
    compile_ms,
    compile_peak,
    cache_size,
    metadata,
):
    full_ids = prefix_ids + suffix_ids
    prefix_length = len(prefix_ids)

    full_logits = full_forward(model, full_ids)
    cached_logits = cached_forward(
        model, frozen, prefix_length, suffix_ids
    )
    metrics = logit_metrics(full_logits, cached_logits)

    full_times, full_peaks = timed_cuda(
        lambda: full_forward(model, full_ids),
        repeats=REPEATS,
        warmups=WARMUPS,
    )

    ideal_values = []
    ideal_peaks = []
    for _ in range(WARMUPS):
        cached_forward(model, frozen, prefix_length, suffix_ids)
    sync()

    for _ in range(REPEATS):
        prepared_cache = fresh_cache(model, frozen)
        device = model_device(model)
        suffix = torch.tensor(
            suffix_ids, dtype=torch.long, device=device
        ).unsqueeze(0)
        total_length = prefix_length + suffix.shape[1]
        attention_mask = torch.ones(
            (1, total_length), dtype=torch.long, device=device
        )
        positions = torch.arange(
            prefix_length, total_length, device=device
        ).unsqueeze(0)
        kwargs = filtered_forward_kwargs(model, {
            "input_ids": suffix,
            "attention_mask": attention_mask,
            "position_ids": positions,
            "cache_position": positions[0],
            "past_key_values": prepared_cache,
            "use_cache": True,
        })

        torch.cuda.reset_peak_memory_stats()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        sync()
        start.record()
        model(**kwargs)
        end.record()
        sync()
        ideal_values.append(float(start.elapsed_time(end)))
        ideal_peaks.append(int(torch.cuda.max_memory_allocated()))

    materialized_times, materialized_peaks = timed_cuda(
        lambda: cached_forward(
            model, frozen, prefix_length, suffix_ids
        ),
        repeats=REPEATS,
        warmups=WARMUPS,
    )

    full_summary = summarize_times(full_times)
    ideal_summary = summarize_times(ideal_values)
    materialized_summary = summarize_times(materialized_times)

    full_ms = full_summary["median_ms"]
    ideal_ms = ideal_summary["median_ms"]
    materialized_ms = materialized_summary["median_ms"]

    ideal_saving = full_ms - ideal_ms
    materialized_saving = full_ms - materialized_ms

    break_even_ideal = (
        compile_ms / ideal_saving if ideal_saving > 0 else float("inf")
    )
    break_even_materialized = (
        compile_ms / materialized_saving
        if materialized_saving > 0
        else float("inf")
    )

    stream_baseline = STREAM_REQUESTS * full_ms
    stream_ideal = compile_ms + STREAM_REQUESTS * ideal_ms
    stream_materialized = (
        compile_ms + STREAM_REQUESTS * materialized_ms
    )

    correctness_pass = (
        metrics["top1_agreement"] == 1.0
        and metrics["cosine_similarity"] > 0.999
        and metrics["kl_full_to_cached"] < 1e-3
    )

    return {
        **metadata,
        "prefix_tokens": prefix_length,
        "suffix_tokens": len(suffix_ids),
        "total_tokens": len(full_ids),
        "compile_ms": compile_ms,
        "cache_bytes": cache_size,
        "compile_peak_allocated_bytes": compile_peak,
        "baseline_median_ms": full_ms,
        "ideal_warm_median_ms": ideal_ms,
        "materialized_warm_median_ms": materialized_ms,
        "ideal_speedup": full_ms / ideal_ms,
        "materialized_speedup": full_ms / materialized_ms,
        "stream_requests": STREAM_REQUESTS,
        "ideal_stream_speedup": stream_baseline / stream_ideal,
        "materialized_stream_speedup": (
            stream_baseline / stream_materialized
        ),
        "ideal_break_even_requests": break_even_ideal,
        "materialized_break_even_requests": break_even_materialized,
        "baseline_peak_allocated_bytes": max(full_peaks),
        "ideal_peak_allocated_bytes": max(ideal_peaks),
        "materialized_peak_allocated_bytes": max(materialized_peaks),
        "correctness_pass": correctness_pass,
        **metrics,
        "baseline_raw_ms": json.dumps(full_times),
        "ideal_raw_ms": json.dumps(ideal_values),
        "materialized_raw_ms": json.dumps(materialized_times),
        "baseline_ci95_low_ms": full_summary["mean_ci95_low_ms"],
        "baseline_ci95_high_ms": full_summary["mean_ci95_high_ms"],
        "ideal_ci95_low_ms": ideal_summary["mean_ci95_low_ms"],
        "ideal_ci95_high_ms": ideal_summary["mean_ci95_high_ms"],
    }

## Model benchmark

Models are loaded and released sequentially to limit GPU memory use. A failure in one model does not stop the remaining runs.

In [ ]:
result_rows = []
failure_rows = []
model_rows = []
safety_rows = []

for spec in MODELS:
    family = spec["family"]
    model_id = spec["model_id"]
    print("\n" + "=" * 80)
    print("MODEL:", family, model_id)
    print("=" * 80)

    tokenizer = None
    model = None

    try:
        clear_gpu()
        tokenizer, model, revision = load_model(spec)
        max_context = maximum_context(model)
        params = parameter_count(model)
        allocated_after_load = int(torch.cuda.memory_allocated())
        reserved_after_load = int(torch.cuda.memory_reserved())

        model_rows.append({
            "family": family,
            "model_id": model_id,
            "revision": revision,
            "parameters": params,
            "quantization": spec["quantization"] or "fp16",
            "max_context": max_context,
            "allocated_after_load_bytes": allocated_after_load,
            "reserved_after_load_bytes": reserved_after_load,
        })

        suffix_texts = SUFFIXES[:CFG["suffixes_per_workload"]]

        for workload in CFG["workloads"]:
            base_text = WORKLOAD_PREFIXES[workload]

            for requested_context in CONTEXTS:
                suffix_token_lists = [
                    encode_suffix(tokenizer, text)
                    for text in suffix_texts
                ]
                maximum_suffix = max(map(len, suffix_token_lists))
                prefix_target = min(
                    requested_context,
                    max_context - maximum_suffix - 1,
                )
                if prefix_target < 32:
                    print("Skipping context beyond model limit.")
                    continue

                prefix_ids = make_exact_prefix(
                    tokenizer, base_text, prefix_target
                )
                key = cache_key(
                    model_id, revision, tokenizer, prefix_ids
                )

                frozen, compile_ms, compile_peak = compile_prefix(
                    model, prefix_ids
                )
                cache_size = cache_bytes(frozen)

                # Planner safety: only an exact token prefix may hit.
                altered = list(prefix_ids)
                altered[len(altered) // 2] = (
                    altered[len(altered) // 2] + 1
                ) % max(2, tokenizer.vocab_size)
                exact_hit = cache_key(
                    model_id, revision, tokenizer, prefix_ids
                ) == key
                altered_hit = cache_key(
                    model_id, revision, tokenizer, altered
                ) == key
                safety_rows.append({
                    "family": family,
                    "model_id": model_id,
                    "workload": workload,
                    "prefix_tokens": len(prefix_ids),
                    "exact_prefix_hit": exact_hit,
                    "one_token_change_hit": altered_hit,
                    "safety_pass": exact_hit and not altered_hit,
                })

                for suffix_index, suffix_ids in enumerate(
                    suffix_token_lists
                ):
                    metadata = {
                        "family": family,
                        "model_id": model_id,
                        "revision": revision,
                        "quantization": spec["quantization"] or "fp16",
                        "workload": workload,
                        "requested_context": requested_context,
                        "suffix_index": suffix_index,
                        "cache_key_prefix": key[:16],
                        "gpu": GPU_NAME,
                        "profile": RUN_PROFILE,
                    }

                    row = benchmark_one_suffix(
                        model=model,
                        tokenizer=tokenizer,
                        frozen=frozen,
                        prefix_ids=prefix_ids,
                        suffix_ids=suffix_ids,
                        compile_ms=compile_ms,
                        compile_peak=compile_peak,
                        cache_size=cache_size,
                        metadata=metadata,
                    )
                    result_rows.append(row)
                    print(
                        family,
                        workload,
                        requested_context,
                        f"ideal={row['ideal_speedup']:.2f}x",
                        f"materialized={row['materialized_speedup']:.2f}x",
                        f"stream={row['materialized_stream_speedup']:.2f}x",
                        "correct=", row["correctness_pass"],
                    )

                del frozen
                clear_gpu()

    except Exception as error:
        failure_rows.append({
            "family": family,
            "model_id": model_id,
            "error_type": type(error).__name__,
            "error": repr(error),
            "traceback": traceback.format_exc(),
        })
        print("FAILED:", family, repr(error))

    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        clear_gpu()

results_df = pd.DataFrame(result_rows)
failures_df = pd.DataFrame(failure_rows)
models_df = pd.DataFrame(model_rows)
safety_df = pd.DataFrame(safety_rows)

results_df.to_csv(RESULTS / "hf_runtime_results.csv", index=False)
failures_df.to_csv(RESULTS / "failures.csv", index=False)
models_df.to_csv(RESULTS / "models.csv", index=False)
safety_df.to_csv(RESULTS / "cache_key_safety.csv", index=False)

display(results_df.head())
display(failures_df)

## Evaluation criteria

A model passes when all correctness and cache-key checks pass, materialized reuse is beneficial on most long-prefix rows, and the median 16-request amortized speedup exceeds 1.05. Ideal and materialized results are reported separately.

In [ ]:
summary_rows = []

if not results_df.empty:
    for family, group in results_df.groupby("family"):
        long_group = group[group["prefix_tokens"] >= 1024]
        if long_group.empty:
            long_group = group

        correctness_rate = float(group["correctness_pass"].mean())
        family_safety = safety_df[safety_df["family"] == family]
        safety_rate = (
            float(family_safety["safety_pass"].mean())
            if not family_safety.empty else 0.0
        )
        positive_materialized_rate = float(
            (long_group["materialized_speedup"] > 1.0).mean()
        )
        median_stream_speedup = float(
            long_group["materialized_stream_speedup"].median()
        )
        median_ideal_stream = float(
            long_group["ideal_stream_speedup"].median()
        )

        consistent_pass = (
            correctness_rate == 1.0
            and safety_rate == 1.0
            and positive_materialized_rate >= 0.5
            and median_stream_speedup > 1.05
        )

        summary_rows.append({
            "family": family,
            "rows": len(group),
            "correctness_rate": correctness_rate,
            "cache_key_safety_rate": safety_rate,
            "long_context_positive_materialized_rate": (
                positive_materialized_rate
            ),
            "median_materialized_stream_speedup": (
                median_stream_speedup
            ),
            "median_ideal_stream_speedup": median_ideal_stream,
            "consistent_gain_pass": consistent_pass,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS / "model_consistency_summary.csv", index=False)

all_requested_families = {m["family"] for m in MODELS}
completed_families = set(summary_df["family"]) if not summary_df.empty else set()
missing_families = sorted(all_requested_families - completed_families)

global_verdict = {
    "requested_families": sorted(all_requested_families),
    "completed_families": sorted(completed_families),
    "missing_families": missing_families,
    "all_models_completed": not missing_families,
    "all_completed_models_consistent": (
        bool(summary_df["consistent_gain_pass"].all())
        if not summary_df.empty else False
    ),
    "global_consistent_gain_pass": (
        not missing_families
        and not summary_df.empty
        and bool(summary_df["consistent_gain_pass"].all())
    ),
}

(RESULTS / "global_verdict.json").write_text(
    json.dumps(global_verdict, indent=2),
    encoding="utf-8",
)

display(summary_df)
print(json.dumps(global_verdict, indent=2))

In [ ]:
if not results_df.empty:
    plot_df = (
        results_df
        .groupby(["family", "prefix_tokens"], as_index=False)
        .agg(
            ideal_speedup=("ideal_speedup", "median"),
            materialized_speedup=("materialized_speedup", "median"),
            stream_speedup=("materialized_stream_speedup", "median"),
        )
    )

    for family, group in plot_df.groupby("family"):
        plt.plot(
            group["prefix_tokens"],
            group["materialized_speedup"],
            marker="o",
            label=family,
        )
    plt.axhline(1.0, linestyle="--")
    plt.xscale("log", base=2)
    plt.xlabel("Exact shared-prefix tokens")
    plt.ylabel("Full prefill / materialized warm latency")
    plt.title("MOSAIC-prefix warm speedup across model families")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURES / "materialized_speedup_by_context.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

    for family, group in plot_df.groupby("family"):
        plt.plot(
            group["prefix_tokens"],
            group["stream_speedup"],
            marker="o",
            label=family,
        )
    plt.axhline(1.0, linestyle="--")
    plt.xscale("log", base=2)
    plt.xlabel("Exact shared-prefix tokens")
    plt.ylabel(f"{STREAM_REQUESTS}-request amortized speedup")
    plt.title("Compile cost included")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURES / "amortized_stream_speedup.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

    cache_plot = (
        results_df
        .groupby(["family", "prefix_tokens"], as_index=False)
        .agg(cache_bytes=("cache_bytes", "median"))
    )
    for family, group in cache_plot.groupby("family"):
        plt.plot(
            group["prefix_tokens"],
            group["cache_bytes"] / 2**20,
            marker="o",
            label=family,
        )
    plt.xscale("log", base=2)
    plt.xlabel("Exact shared-prefix tokens")
    plt.ylabel("Frozen KV cache size (MiB)")
    plt.title("Persistent cache memory cost")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURES / "cache_memory_by_context.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

## Negative control

A one-token prefix change must produce a cache miss. An identical substring inside a different causal prefix is not reusable.

In [ ]:
if not safety_df.empty:
    assert safety_df["exact_prefix_hit"].all()
    assert (~safety_df["one_token_change_hit"]).all()
    assert safety_df["safety_pass"].all()
    print("PASS: all one-token prefix mutations were rejected.")
    display(safety_df.groupby("family")["safety_pass"].mean())

## Optional vLLM APC baseline

When `RUN_VLLM=True`, the notebook compares the same shared-prefix workload with vLLM Automatic Prefix Caching in a separate subprocess.

In [ ]:
VLLM_MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
]

if RUN_VLLM:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "vllm>=0.11,<1",
    ])

    vllm_script = ROOT / "run_vllm_apc.py"
    vllm_script.write_text(r'''
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

model_id = sys.argv[1]
output_csv = Path(sys.argv[2])
token = os.environ.get("HF_TOKEN")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)

shared = (
    "You are a careful analysis agent. Preserve exact names and do not invent results. "
    "The following repository specification is static across requests. "
) * 700
suffixes = [
    "\nUser: Explain cache invalidation.",
    "\nUser: Propose branch isolation tests.",
    "\nUser: Estimate the break-even request count.",
    "\nUser: Identify an unsafe cache key.",
    "\nUser: Compare cold and warm execution.",
    "\nUser: Summarize the runtime limitations.",
]
prompts = [shared + suffix for suffix in suffixes]
sampling = SamplingParams(temperature=0.0, max_tokens=1)

rows = []
for enabled in (False, True):
    engine = LLM(
        model=model_id,
        dtype="float16",
        enable_prefix_caching=enabled,
        gpu_memory_utilization=0.85,
        max_model_len=4096,
    )
    engine.generate([prompts[0]], sampling, use_tqdm=False)

    for round_index in range(4):
        start = time.perf_counter()
        outputs = engine.generate(prompts, sampling, use_tqdm=False)
        elapsed = (time.perf_counter() - start) * 1000
        rows.append({
            "model_id": model_id,
            "enable_prefix_caching": enabled,
            "round": round_index,
            "requests": len(prompts),
            "elapsed_ms": elapsed,
            "requests_per_s": len(prompts) / (elapsed / 1000),
            "output_token_ids": json.dumps([
                output.outputs[0].token_ids for output in outputs
            ]),
        })
    del engine

pd.DataFrame(rows).to_csv(output_csv, index=False)
''', encoding="utf-8")

    vllm_outputs = []
    for model_id in VLLM_MODELS:
        target = RESULTS / (
            "vllm_apc_" + model_id.replace("/", "__") + ".csv"
        )
        env = dict(os.environ)
        if HF_TOKEN:
            env["HF_TOKEN"] = HF_TOKEN
        subprocess.run(
            [sys.executable, str(vllm_script), model_id, str(target)],
            check=True,
            env=env,
        )
        vllm_outputs.append(pd.read_csv(target))

    vllm_df = pd.concat(vllm_outputs, ignore_index=True)
    vllm_df.to_csv(RESULTS / "vllm_apc_results.csv", index=False)
    display(vllm_df)

    vllm_summary = (
        vllm_df
        .groupby(["model_id", "enable_prefix_caching"], as_index=False)
        .agg(
            median_elapsed_ms=("elapsed_ms", "median"),
            median_requests_per_s=("requests_per_s", "median"),
        )
    )
    display(vllm_summary)
else:
    print("RUN_VLLM=False: optional strong baseline skipped.")

## Tables and result archive

In [ ]:
environment = {
    "completed_utc": pd.Timestamp.utcnow().isoformat(),
    "run_profile": RUN_PROFILE,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": GPU_NAME,
    "gpu_memory_bytes": GPU_BYTES,
    "repeats": REPEATS,
    "warmups": WARMUPS,
    "stream_requests": STREAM_REQUESTS,
    "models_requested": MODELS,
    "run_vllm": RUN_VLLM,
}
(RESULTS / "environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

if not summary_df.empty:
    latex_table = summary_df.to_latex(
        index=False,
        float_format=lambda x: f"{x:.3f}",
        caption=(
            "Cross-model consistency of exact shared-prefix "
            "runtime gains."
        ),
        label="tab:mosaic_llm_consistency",
    )
    (RESULTS / "model_consistency_summary.tex").write_text(
        latex_table,
        encoding="utf-8",
    )

readme = f'''# MOSAIC LLM Runtime Benchmark Results

Profile: {RUN_PROFILE}
GPU: {GPU_NAME}
Requested models: {len(MODELS)}
Completed families: {len(completed_families)}
Missing families: {", ".join(missing_families) if missing_families else "none"}
Global consistent-gain pass: {global_verdict["global_consistent_gain_pass"]}

Interpretation:
- ideal speedup excludes cache branch materialization;
- materialized speedup includes Hugging Face cache reconstruction;
- stream speedup includes one prefix compilation over {STREAM_REQUESTS} requests;
- this benchmark validates exact shared-prefix reuse, not arbitrary internal substring reuse;
- compare with vLLM APC before claiming novelty.
'''
(ROOT / "README.md").write_text(readme, encoding="utf-8")

zip_path = Path(
    "/kaggle/working/mosaic_llm_runtime_consistency_results.zip"
)
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in ROOT.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(ROOT))

print("Result ZIP:", zip_path)
print("Upload the executed notebook and the ZIP.")